# Beam-angle selection with LR-QAOA (synthetic tutorial)

**Clinical story (simplified):** radiation therapy plans choose a small subset of gantry angles. Each angle contributes dose to tumor and organs-at-risk (OAR). We want **enough tumor coverage** while keeping **OAR exposure low**, with a fixed number of beams.

This notebook uses a **toy dose-influence matrix** (no patient CT/HDF5) to show the Haiqu pipeline:

1. Build a cardinality QUBO as an `OptimizationProblem`.
2. Brute-force check the optimum on 6 angles, pick 2.
3. Run `build_lr_qaoa_circuit` → `run` → `postprocess`.
4. Read the measurement distribution (expected cost and CVaR).

Related: [DikshantDulal/beam-angle-radiotherapy-optimization](https://github.com/DikshantDulal/beam-angle-radiotherapy-optimization).

### Encountered an issue? Let us know

Found a bug or have a question? [Submit feedback](https://feedback.haiqu.ai/)

## 1. Synthetic influence data

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
from qiskit_addon_opt_mapper.problems import OptimizationProblem

from haiqu.sdk import haiqu, set_haiqu_mpl_style
from haiqu.sdk.optimization import cvar_expectation, evaluate_problem_cost, validate_optimization_problem

HAIQU = {
    "black": "#000000",
    "white": "#FFFFFF",
    "grey": "#CCCCCC",
    "morpho_blue": "#093188",
    "light_blue": "#C4D5FF",
    "red": "#8D2816",
    "orange": "#FEA450",
}

set_haiqu_mpl_style()

beam_names = [f"G{i * 30:03d}" for i in range(6)]
influence = np.array(
    [
        [0.85, 0.75, 0.35, 0.55, 0.25],
        [0.70, 0.80, 0.45, 0.30, 0.40],
        [0.60, 0.55, 0.20, 0.50, 0.35],
        [0.75, 0.65, 0.40, 0.25, 0.45],
        [0.50, 0.70, 0.15, 0.35, 0.30],
        [0.65, 0.60, 0.30, 0.45, 0.20],
    ]
)
tumor_idx = [0, 1]
oar_idx = [2, 3, 4]

n_beams = len(beam_names)
n_select = 2
penalty = 12.0

print("Beams:", ", ".join(beam_names))
print(influence)

## 2. Build an unconstrained QUBO (objective + cardinality penalty)

In [ ]:
linear = {}
quadratic = {}
for i in range(n_beams):
    tumor_dose = influence[i, tumor_idx].sum()
    oar_dose = influence[i, oar_idx].sum()
    linear[i] = float(oar_dose - 0.6 * tumor_dose) + penalty * (1 - 2 * n_select)
for i in range(n_beams):
    for j in range(i + 1, n_beams):
        quadratic[(i, j)] = 2 * penalty

problem = OptimizationProblem("beam_cardinality")
problem.binary_var_list(n_beams)
problem.minimize(linear=linear, quadratic=quadratic)
validate_optimization_problem(problem)
print(problem.prettyprint())

## 3. Brute-force optimum

In [ ]:
def bitstring_from_selection(selected):
    chars = ["0"] * n_beams
    for qubit in selected:
        chars[n_beams - 1 - qubit] = "1"
    return "".join(chars)

brute = []
for combo in itertools.combinations(range(n_beams), n_select):
    bs = bitstring_from_selection(combo)
    brute.append((evaluate_problem_cost(problem, bs), bs, combo))
brute.sort(key=lambda x: x[0])

optimal_cost, optimal_bs, optimal_combo = brute[0]
print(f"Optimal cost: {optimal_cost:.3f}")
print(f"Optimal beams: {[beam_names[i] for i in optimal_combo]}")

## 4. Haiqu LR-QAOA pipeline

Set `HAIQU_API_KEY` or pass `api_access_key=` to `haiqu.login()`.

In [ ]:
haiqu.login()
haiqu.init("Beam Angle LR-QAOA Tutorial")

p = 2
shots = 1000
circuit = haiqu.build_lr_qaoa_circuit(problem, p=p)
job = haiqu.run(circuit, device_id="aer_simulator", shots=shots)
raw_counts = job.result()[0]

In [ ]:
def _is_minimize(problem):
    return "MIN" in problem.objective.sense.name

def best_cost(problem, costs):
    values = list(costs.values())
    return min(values) if _is_minimize(problem) else max(values)

def costs_from_counts(problem, counts):
    return {bitstring: evaluate_problem_cost(problem, bitstring) for bitstring in counts}

def expected_cost(problem, counts):
    total = sum(counts.values())
    if total <= 0:
        return float("nan")
    return sum((weight / total) * evaluate_problem_cost(problem, bitstring) for bitstring, weight in counts.items())

raw_costs = costs_from_counts(problem, raw_counts)
print(f"Raw best cost: {best_cost(problem, raw_costs):.3f} (optimal {optimal_cost:.3f})")
print(f"Raw expected cost: {expected_cost(problem, raw_counts):.3f}")

processed_costs, processed_counts = haiqu.postprocess(
    counts=raw_counts,
    problem=problem,
    postprocess_iterations=5,
    seed=42,
)
print(f"Post-processed best cost: {best_cost(problem, processed_costs):.3f}")
print(f"Post-processed expected cost: {expected_cost(problem, processed_counts):.3f}")

print(f"CVaR raw: {cvar_expectation(raw_counts, problem=problem, alpha=0.15):.3f}")
print(f"CVaR post-processed: {cvar_expectation(processed_counts, problem=problem, alpha=0.15):.3f}")

## 5. Plot top bitstrings

In [ ]:
labels = []
values = []
for bitstring, weight in sorted(raw_counts.items(), key=lambda kv: -kv[1])[:8]:
    active = [beam_names[i] for i in range(n_beams) if bitstring[n_beams - 1 - i] == "1"]
    labels.append("+".join(active) if active else "none")
    values.append(weight)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(labels, values, color=HAIQU["grey"], edgecolor=HAIQU["black"], linewidth=0.6)
ax.set_ylabel("Probability mass (top 8 strings)")
ax.set_xlabel("Selected beams")
ax.set_title("Raw LR-QAOA measurement support")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 6. Limitations

- Toy influences only; real plans need deliverability and anatomy.
- Tune `penalty` when scaling cardinality constraints.
- Prefer the explicit pipeline over deprecated `haiqu.solve_qubo`.